In [2]:
# Cell 1  default Kaggle starter cell

# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/sample_submission.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/train.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_49.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_90.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_77.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_66.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_54.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_42.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_81.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_72.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final

In [3]:

# Cell 2: setup - load data lists and check GPU
import torch
from tqdm.auto import tqdm

BASE = '/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final'
train = pd.read_csv(BASE + '/train.csv')
test = pd.read_csv(BASE + '/test.csv')

jobs = ([(f, 'train') for f in train['filename']] +
        [(f, 'test') for f in test['filename']])
n_train = len(train)
y = train['label'].values

print(train.shape, test.shape, len(jobs), "GPU:", torch.cuda.is_available())




(769, 2) (216, 2) 985 GPU: True


In [6]:
# Cell 4 whsiper downlaod

!pip install -q openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 24.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [7]:
# Cell 5: check Whisper is available
try:
    import whisper
    print("whisper ok")
except Exception as e:
    print("FAILED:", type(e).__name__, e)

whisper ok


In [8]:
# Cell 6 

# keep the time sequence instead of averaging it away
model = whisper.load_model("small")
LAYERS = (8, 12)

def get_frames(path):
    audio = whisper.load_audio(path)
    out = np.zeros((len(LAYERS), 150, 768), dtype=np.float16)   # 2 halves x 75 frames
    store, hooks = {}, []
    for i in LAYERS:
        hooks.append(model.encoder.blocks[i - 1].register_forward_hook(
            lambda m, inp, o, i=i: store.__setitem__(i, o)))
    valid = 0
    for k, start in enumerate((0, 480000)):
        ch = audio[start:start + 480000]
        if len(ch) < 16000:
            continue
        valid += min(75, int(np.ceil(len(ch) / 480000 * 75)))
        mel = whisper.log_mel_spectrogram(whisper.pad_or_trim(ch)).to(model.device)
        with torch.no_grad():
            model.encoder(mel.unsqueeze(0).half())
        for j, i in enumerate(LAYERS):
            x = store[i].squeeze(0).float()                  # (1500, 768)
            x = x.reshape(75, 20, 768).mean(dim=1)           # shrink time: 1500 -> 75
            out[j, k * 75:(k + 1) * 75] = x.cpu().numpy().astype(np.float16)
    for h in hooks:
        h.remove()
    return out, valid

frames, valid = [], []
for fname, folder in tqdm(jobs):
    f, v = get_frames(BASE + '/' + folder + '/' + fname)
    frames.append(f); valid.append(v)

frames = np.stack(frames)          # (985, 2, 150, 768)
valid = np.array(valid)
np.save('/kaggle/working/frames_small.npy', frames)
np.save('/kaggle/working/frames_valid.npy', valid)
print(frames.shape, valid.min(), valid.max())

100%|████████████████████████████████████████| 461M/461M [00:01<00:00, 348MiB/s]


  0%|          | 0/985 [00:00<?, ?it/s]

(985, 2, 150, 768) 17 150


In [20]:
# Cell 9: load Whisper layers 4, 8, 12 and check the shapes
P = '/kaggle/input/datasets/komalsaini299/layer4812/'
layer_embs = {i: np.load(P + f'whisper_layer{i}.npy') for i in (4, 8, 12)}
X_audio = np.hstack([layer_embs[4], layer_embs[8], layer_embs[12]])
print({i: e.shape for i, e in layer_embs.items()}, X_audio.shape)

{4: (985, 768), 8: (985, 768), 12: (985, 768)} (985, 2304)


In [21]:
# Cell 10: Ridge cross-validation on Whisper layers 4+8+12
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_predict

kf = KFold(n_splits=5, shuffle=True, random_state=42)
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1000))
pred = np.clip(cross_val_predict(ridge, X_audio[:n_train], y, cv=kf), 0, 5)
print("Ridge CV RMSE:", round(float(np.sqrt(((pred - y) ** 2).mean())), 3))

Ridge CV RMSE: 0.556


In [23]:
# Cell 12: WavLM mean+std embeddings (about 12 min)
import soundfile as sf
from transformers import AutoFeatureExtractor, WavLMModel

fe = AutoFeatureExtractor.from_pretrained('microsoft/wavlm-base-plus')
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus').to('cuda').eval()

def wavlm_embed_ms(path, layers=(4, 8, 12)):
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        import torchaudio
        audio = torchaudio.functional.resample(
            torch.tensor(audio, dtype=torch.float32), sr, 16000).numpy()
    audio = audio[:16000 * 60].astype('float32')
    inputs = fe(audio, sampling_rate=16000, return_tensors='pt')
    with torch.no_grad():
        out = wavlm(inputs['input_values'].to('cuda'), output_hidden_states=True)
    parts = []
    for i in layers:
        h = out.hidden_states[i].squeeze(0)
        parts += [h.mean(dim=0).cpu().numpy(), h.std(dim=0).cpu().numpy()]
    return np.concatenate(parts)

wav_ms = np.vstack([wavlm_embed_ms(BASE + '/' + folder + '/' + fname)
                    for fname, folder in tqdm(jobs)])
np.save('/kaggle/working/wavlm_ms.npy', wav_ms)
print(wav_ms.shape)

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

  0%|          | 0/985 [00:00<?, ?it/s]

(985, 4608)


In [24]:
# Cell 13: Ridge + SVR on Whisper layers + WavLM mean/std
from sklearn.svm import SVR
from sklearn.decomposition import PCA

X_best = np.hstack([X_audio, wav_ms])
print(X_best.shape)

ridge = make_pipeline(StandardScaler(), Ridge(alpha=1000))
svr = make_pipeline(StandardScaler(), PCA(n_components=200, random_state=42), SVR(C=10, kernel='rbf'))

p_ridge = np.clip(cross_val_predict(ridge, X_best[:n_train], y, cv=kf), 0, 5)
p_svr = np.clip(cross_val_predict(svr, X_best[:n_train], y, cv=kf), 0, 5)
p_blend = 0.3 * p_ridge + 0.7 * p_svr

def rmse(p):
    return round(float(np.sqrt(((p - y) ** 2).mean())), 3)

print("Ridge:", rmse(p_ridge), "| SVR:", rmse(p_svr), "| Blend:", rmse(p_blend))

(985, 6912)
Ridge: 0.537 | SVR: 0.517 | Blend: 0.513


In [27]:
# Cell 16: WavLM LARGE mean+std embeddings (about 25-30 min)
from transformers import AutoFeatureExtractor, WavLMModel

fe_l = AutoFeatureExtractor.from_pretrained('microsoft/wavlm-large')
wavlm_l = WavLMModel.from_pretrained('microsoft/wavlm-large').to('cuda').eval()

def wavlm_large_embed(path, layers=(6, 12, 18, 24)):
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        import torchaudio
        audio = torchaudio.functional.resample(
            torch.tensor(audio, dtype=torch.float32), sr, 16000).numpy()
    audio = audio[:16000 * 60].astype('float32')
    inputs = fe_l(audio, sampling_rate=16000, return_tensors='pt')
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        out = wavlm_l(inputs['input_values'].to('cuda'), output_hidden_states=True)
    parts = []
    for i in layers:
        h = out.hidden_states[i].squeeze(0).float()
        parts += [h.mean(dim=0).cpu().numpy(), h.std(dim=0).cpu().numpy()]
    return np.concatenate(parts)

wavl_ms = np.vstack([wavlm_large_embed(BASE + '/' + folder + '/' + fname)
                     for fname, folder in tqdm(jobs)])
np.save('/kaggle/working/wavlm_large_ms.npy', wavl_ms)
print(wavl_ms.shape)

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.22k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.26GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            

  0%|          | 0/985 [00:00<?, ?it/s]

(985, 8192)


In [28]:
# Cell 17: does WavLM large help?
sets = {
    'large only': wavl_ms,
    'whisper + base + large': np.hstack([X_best, wavl_ms]),
}
for name, e in sets.items():
    for a in [1000, 3000]:
        r = make_pipeline(StandardScaler(), Ridge(alpha=a))
        p = np.clip(cross_val_predict(r, e[:n_train], y, cv=kf), 0, 5)
        print(name, "| ridge alpha", a, "| RMSE:", rmse(p))

large only | ridge alpha 1000 | RMSE: 0.521
large only | ridge alpha 3000 | RMSE: 0.531
whisper + base + large | ridge alpha 1000 | RMSE: 0.515
whisper + base + large | ridge alpha 3000 | RMSE: 0.518


In [29]:
# Cell 18: Ridge + SVR blend on Whisper + WavLM base + WavLM large
X_all = np.hstack([X_best, wavl_ms])
print(X_all.shape)

ridge = make_pipeline(StandardScaler(), Ridge(alpha=1000))
p_ridge = np.clip(cross_val_predict(ridge, X_all[:n_train], y, cv=kf), 0, 5)

for C in [3, 10]:
    svr = make_pipeline(StandardScaler(), PCA(n_components=200, random_state=42), SVR(C=C, kernel='rbf'))
    p_svr = np.clip(cross_val_predict(svr, X_all[:n_train], y, cv=kf), 0, 5)
    print("SVR C", C, "|", rmse(p_svr),
          "| blend 0.3/0.7:", rmse(0.3 * p_ridge + 0.7 * p_svr),
          "| blend 0.5/0.5:", rmse(0.5 * p_ridge + 0.5 * p_svr))

(985, 15104)
SVR C 3 | 0.518 | blend 0.3/0.7: 0.508 | blend 0.5/0.5: 0.506
SVR C 10 | 0.514 | blend 0.3/0.7: 0.505 | blend 0.5/0.5: 0.503


In [30]:
# Cell 19: train on all 769 files, predict test, write submission
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1000))
svr = make_pipeline(StandardScaler(), PCA(n_components=200, random_state=42), SVR(C=10, kernel='rbf'))

ridge.fit(X_all[:n_train], y)
svr.fit(X_all[:n_train], y)

test_pred5 = np.clip(0.5 * ridge.predict(X_all[n_train:]) + 0.5 * svr.predict(X_all[n_train:]), 0, 5)

sub5 = pd.DataFrame({'filename': test['filename'].values, 'label': test_pred5})
print(sub5.shape, sub5['label'].isna().sum())
sub5.to_csv('/kaggle/working/submission.csv', index=False)
sub5.head(3)

(216, 2) 0


,filename,label
0,audio_128.wav,2.828134
1,audio_156.wav,4.240130
2,audio_19.wav,4.602167


In [31]:
# Cell 20: compare test predictions with out-of-fold train predictions
print("train true  : mean", round(float(y.mean()), 2), "std", round(float(y.std()), 2))
print("train pred  : mean", round(float(p_blend.mean()), 2), "std", round(float(p_blend.std()), 2))
print("test pred   : mean", round(float(test_pred5.mean()), 2), "std", round(float(test_pred5.std()), 2))
print("test preds below 1.0:", int((test_pred5 < 1.0).sum()), "of", len(test_pred5))
print("train true zeros:", int((y == 0).sum()), "of", len(y))

train true  : mean 3.31 std 1.24
train pred  : mean 3.35 std 1.11
test pred   : mean 3.32 std 0.83
test preds below 1.0: 0 of 216
train true zeros: 37 of 769


In [33]:
# Cell 21: CV error on non-zero files only (closer to the test set)
nz = y > 0
print("Blend, all files   :", rmse(p_blend))
print("Blend, non-zero only:", round(float(np.sqrt(((p_blend[nz] - y[nz]) ** 2).mean())), 3))
print("Std of true scores, non-zero only:", round(float(y[nz].std()), 2))

Blend, all files   : 0.513
Blend, non-zero only: 0.524
Std of true scores, non-zero only: 1.01


In [35]:
# Cell 22: Whisper MEDIUM encoder, mean+std over time, layers 6/12/18/24
wm = whisper.load_model("medium")
LAY = (6, 12, 18, 24)

def whisper_ms(path):
    audio = whisper.load_audio(path)
    store, hooks = {}, []
    for i in LAY:
        hooks.append(wm.encoder.blocks[i - 1].register_forward_hook(
            lambda m, inp, o, i=i: store.__setitem__(i, o)))
    means = {i: [] for i in LAY}
    stds = {i: [] for i in LAY}
    for start in (0, 480000):
        ch = audio[start:start + 480000]
        if len(ch) < 16000:
            continue
        n = min(1500, int(np.ceil(len(ch) / 320)))      # real (non-padded) frames
        mel = whisper.log_mel_spectrogram(whisper.pad_or_trim(ch)).to(wm.device)
        with torch.no_grad():
            wm.encoder(mel.unsqueeze(0).half())
        for i in LAY:
            h = store[i].squeeze(0).float()[:n]
            means[i].append(h.mean(0).cpu().numpy())
            stds[i].append(h.std(0).cpu().numpy())
    for h in hooks:
        h.remove()
    return np.concatenate([np.concatenate([np.mean(means[i], 0), np.mean(stds[i], 0)])
                           for i in LAY])

wmed = np.vstack([whisper_ms(BASE + '/' + folder + '/' + fname)
                  for fname, folder in tqdm(jobs)])
np.save('/kaggle/working/whisper_medium_ms.npy', wmed)
print(wmed.shape)

  0%|          | 0/985 [00:00<?, ?it/s]

(985, 8192)


In [36]:
# Cell 23: does Whisper medium add anything?
sets = {
    'medium only': wmed,
    'all four encoders': np.hstack([X_all, wmed]),
}
for name, e in sets.items():
    for a in [1000, 3000]:
        r = make_pipeline(StandardScaler(), Ridge(alpha=a))
        p = np.clip(cross_val_predict(r, e[:n_train], y, cv=kf), 0, 5)
        print(name, "| ridge alpha", a, "| RMSE:", rmse(p))

medium only | ridge alpha 1000 | RMSE: 0.527
medium only | ridge alpha 3000 | RMSE: 0.536
all four encoders | ridge alpha 1000 | RMSE: 0.509
all four encoders | ridge alpha 3000 | RMSE: 0.51


In [37]:
# Cell 24: Ridge + SVR blend on all four encoders, then write submission
X_4 = np.hstack([X_all, wmed])
print(X_4.shape)

ridge = make_pipeline(StandardScaler(), Ridge(alpha=1000))
svr = make_pipeline(StandardScaler(), PCA(n_components=200, random_state=42), SVR(C=10, kernel='rbf'))

p_r = np.clip(cross_val_predict(ridge, X_4[:n_train], y, cv=kf), 0, 5)
p_s = np.clip(cross_val_predict(svr, X_4[:n_train], y, cv=kf), 0, 5)
print("Ridge:", rmse(p_r), "| SVR:", rmse(p_s), "| Blend 0.5/0.5:", rmse(0.5 * p_r + 0.5 * p_s))

ridge.fit(X_4[:n_train], y)
svr.fit(X_4[:n_train], y)
test_pred6 = np.clip(0.5 * ridge.predict(X_4[n_train:]) + 0.5 * svr.predict(X_4[n_train:]), 0, 5)

sub6 = pd.DataFrame({'filename': test['filename'].values, 'label': test_pred6})
print(sub6.shape, sub6['label'].isna().sum())
sub6.to_csv('/kaggle/working/submission.csv', index=False)
sub6.head(3)

(985, 23296)
Ridge: 0.509 | SVR: 0.507 | Blend 0.5/0.5: 0.493
(216, 2) 0


,filename,label
0,audio_128.wav,2.859348
1,audio_156.wav,4.330004
2,audio_19.wav,4.716831
